# Proyecto Aduana BI — Introducción y Arquitectura

---

## ¿Qué hace este proyecto?

Este proyecto construye un **Data Warehouse (DW)** con datos de despachos aduaneros.

Toma datos en bruto de un archivo Excel (con información de importaciones y exportaciones),
los transforma y los organiza en un modelo dimensional para luego analizarlos en **Power BI**.

---

## Conceptos clave que vas a necesitar

### 1. ETL (Extract, Transform, Load)
Es el proceso de:
- **Extraer** datos de una fuente (Excel, CSV, base de datos)
- **Transformar** esos datos (limpiar, normalizar, calcular)
- **Cargar** los datos transformados en un destino (Data Warehouse)

### 2. Data Warehouse (DW)
Es una base de datos diseñada para **análisis y reportes**, no para operaciones del día a día.
Almacena datos históricos organizados para consultas rápidas y eficientes.

### 3. Modelo Dimensional (Star Schema)
Forma de organizar los datos en un DW con dos tipos de tablas:
- **Tabla de Hechos (Fact Table)**: guarda los eventos o transacciones con sus medidas numéricas
- **Tablas de Dimensiones (Dim Tables)**: guardan los atributos descriptivos de cada hecho

### 4. DuckDB
Base de datos SQL embebida en un solo archivo `.duckdb`.
No necesita servidor — es perfecta para análisis local con Python.

### 5. Capas de datos
| Capa | Nombre | Descripción |
|------|--------|-------------|
| 1 | **Bronze** | Datos crudos tal como vienen (Excel, CSV) |
| 2 | **Silver** | Datos limpios y normalizados (Staging) |
| 3 | **Gold** | Datos listos para análisis (Dimensiones + Fact) |


---

## Arquitectura del proyecto

```
┌─────────────────────────────────────────────────────────────────┐
│                        FUENTES (Bronze)                         │
│  DatosXLSX.xlsx              LISTADO_DE_DESTINACIONES.xlsx      │
│  (datos de despachos)        (catálogo de códigos aduaneros)    │
└──────────────────────────────┬──────────────────────────────────┘
                               │  ETL-Cargar-Staging.py
                               ▼
┌─────────────────────────────────────────────────────────────────┐
│                        STAGING (Silver)                         │
│  dw.stg_aduana               dw.stg_destinaciones              │
│  (41 columnas, datos         (catálogo de destinaciones)        │
│   normalizados)                                                 │
└──────────────────────────────┬──────────────────────────────────┘
                               │  ETL-Dimension.py
                               ▼
┌─────────────────────────────────────────────────────────────────┐
│                      DIMENSIONES (Gold)                         │
│  dim_operacion    dim_destinacion   dim_regimen   dim_aduana    │
│  dim_pais         dim_producto      dim_canal     dim_fecha     │
│  dim_medio_transporte  dim_unidad_medida  dim_acuerdo  dim_marca│
└──────────────────────────────┬──────────────────────────────────┘
                               │  ETL-FACT.py
                               ▼
┌─────────────────────────────────────────────────────────────────┐
│                      FACT TABLE (Gold)                          │
│  fact_aduana_item                                               │
│  (15 claves foráneas + 19 métricas numéricas)                  │
└──────────────────────────────┬──────────────────────────────────┘
                               │  Power BI (ODBC)
                               ▼
┌─────────────────────────────────────────────────────────────────┐
│                      VISUALIZACIÓN                              │
│  Dash.pbix — Dashboard Power BI                                 │
└─────────────────────────────────────────────────────────────────┘
```


---

## Modelo Star Schema del proyecto

```
                     dim_operacion
                     dim_destinacion
                     dim_regimen
                     dim_aduana
                     dim_pais (origen)
dim_fecha ────────►  fact_aduana_item  ◄──── dim_pais (destino)
                     dim_producto
                     dim_canal
                     dim_medio_transporte
                     dim_unidad_medida
                     dim_acuerdo
                     dim_marca
```

**Grano de la Fact Table:** una fila = una línea de ítem dentro de un despacho aduanero.

**Clave natural:** `(despacho_cifrado, item)` — identifica unívocamente cada línea.


---

## Archivos del proyecto

| Archivo | Rol | Orden de ejecución |
|---------|-----|--------------------|
| `sql/CrearTablas.py` | Crea el esquema y todas las tablas vacías | 1 |
| `etl/ETL-Cargar-Staging.py` | Carga datos del Excel a la capa Staging | 2 |
| `etl/ETL-Dimension.py` | Genera todas las tablas de dimensiones | 3 |
| `etl/ETL-FACT.py` | Carga la tabla de hechos con todos los JOINs | 4 |
| `etl/ConsAUX.py` | Valida la integridad del DW construido | 5 |
| `etl/ProcesoBase.py` | Orquestador: ejecuta los 5 pasos anteriores | 0 (ejecuta todo) |

---

## Tecnologías utilizadas

- **Python 3.12** — lenguaje de programación principal
- **DuckDB** — base de datos SQL embebida (archivo `aduana.duckdb`)
- **Pandas** — lectura de archivos Excel y manipulación de datos
- **Power BI** — visualización final del DW
- **ODBC** — conector entre DuckDB y Power BI


In [ ]:
# Verificar que las librerías necesarias estén instaladas
import importlib

libs = ['duckdb', 'pandas', 'openpyxl']
for lib in libs:
    estado = 'OK' if importlib.util.find_spec(lib) else 'FALTA — instalar con: pip install ' + lib
    print(f'{lib}: {estado}')

: 